# 06 综合案例：智能对话助手（整合优化版）

这份 Notebook 面向“可实操学习”，不仅给出 API 用法，还会解释为什么这样设计。

学习建议：
1. 先顺序跑完所有代码，确认每个阶段输入输出。
2. 再改参数做对比实验（尤其是 chunk_size、k、search_type）。
3. 最后把你自己的数据替换进来，验证迁移效果。

## 学习目标与方法

本章每个文件都采用同一套学习框架：
- 概念：这个模块在 RAG 链路里解决什么问题
- 接口：核心函数、关键参数、常见坑
- 实战：可复用代码模板 + 结果验证
- 迭代：如何优化质量、成本与稳定性

In [ ]:
from __future__ import annotations

# ========= 通用环境初始化 =========
# 说明：
# 1) 读取 .env（如果存在）
# 2) 统一工作目录与资源目录
# 3) 打印关键密钥状态，避免后面运行时报错才发现没配环境

import os
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional

try:
    import dotenv
    dotenv.load_dotenv()
except Exception:
    # 没安装 python-dotenv 也不影响运行，只是不会自动加载 .env
    pass

BASE_DIR = Path.cwd()
ASSET_DIR = BASE_DIR / 'asset'
LOAD_DIR = ASSET_DIR / 'load'
assert LOAD_DIR.exists(), f'未找到数据目录: {LOAD_DIR}'

print('workdir:', BASE_DIR)
print('OPENAI_API_KEY exists:', bool(os.getenv('OPENAI_API_KEY')))
print('TAVILY_API_KEY exists:', bool(os.getenv('TAVILY_API_KEY')))

# 06 综合案例：智能对话助手（优化增强版）

这个版本在原课件流程上做了系统化增强，目标是让你不仅能跑通，还能用于后续项目扩展。

主要增强点：
- 修复编码与可读性问题，完整中文讲解
- 函数化封装：配置、工具、Agent、记忆、评估
- 增加降级逻辑：缺少 API Key 时不崩溃
- 增加可观测与调试：检索命中预览、中间步骤输出
- 增加代码量：批量问答、简单评估、缓存与重试示例

## 1. 环境与配置

建议先在 `.env` 配置：
- `OPENAI_API_KEY`
- `TAVILY_API_KEY`（可选，用于联网搜索）

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

try:
    import dotenv
    dotenv.load_dotenv()
except Exception:
    pass

# 避免部分环境下 WebBaseLoader 对 UA 的告警
os.environ.setdefault('USER_AGENT', 'langchain-rag-agent-demo/1.0')

BASE_DIR = Path.cwd()
ASSET_DIR = BASE_DIR / 'asset'

@dataclass
class AssistantConfig:
    """对话助手配置。

    参数说明：
    - wiki_url: 作为知识库来源的网页链接。
    - embedding_model/chat_model: 向量模型与对话模型。
    - chunk_size/chunk_overlap: 文本切块参数。
    - retriever_k: 检索 top-k 文档数。
    - tavily_max_results: Web 搜索结果条数。
    - temperature: LLM 温度，知识问答建议低温。
    """

    wiki_url: str = 'https://zh.wikipedia.org/wiki/%E7%8C%AB'
    embedding_model: str = 'text-embedding-3-small'
    chat_model: str = 'gpt-4o-mini'
    chunk_size: int = 900
    chunk_overlap: int = 140
    retriever_k: int = 4
    tavily_max_results: int = 3
    temperature: float = 0.1

cfg = AssistantConfig()
print('OPENAI_API_KEY exists:', bool(os.getenv('OPENAI_API_KEY')))
print('TAVILY_API_KEY exists:', bool(os.getenv('TAVILY_API_KEY')))
cfg

## 2. 构建工具：Web Search + Wiki Retriever

工具路由策略：
1. 实时信息问题优先走 Web Search。
2. 领域知识问题优先走 Retriever Tool。
3. 两类工具统一交给 Agent 自动选择。

In [ ]:
from __future__ import annotations

import os
from typing import Any

def build_web_search_tool(*, max_results: int = 3) -> Any:
    """构建联网搜索工具。

    Args:
        max_results: 最大返回条数，值越大召回越广但噪声更高。

    Returns:
        有 Tavily Key 时返回 TavilySearchResults；否则返回 fallback 工具。
    """
    if os.getenv('TAVILY_API_KEY'):
        from langchain_community.tools.tavily_search import TavilySearchResults
        return TavilySearchResults(max_results=max_results)

    from langchain_core.tools import tool

    @tool('web_search_fallback')
    def fallback(query: str) -> str:
        """缺少 TAVILY_API_KEY 时的降级工具。"""
        return f'未配置 TAVILY_API_KEY，无法联网搜索。query={query}'

    return fallback

def get_embeddings(*, model: str) -> Any:
    """获取 Embeddings。

    Args:
        model: embedding 模型名。

    Returns:
        Embeddings 实例。无 OpenAI Key 时回退 FakeEmbeddings（仅演示流程）。
    """
    if os.getenv('OPENAI_API_KEY'):
        from langchain_openai import OpenAIEmbeddings
        return OpenAIEmbeddings(model=model)
    from langchain_core.embeddings import FakeEmbeddings
    print('缺少 OPENAI_API_KEY，回退 FakeEmbeddings。')
    return FakeEmbeddings(size=1536)

def build_wiki_retriever_tool(
    *,
    wiki_url: str,
    embedding_model: str,
    chunk_size: int,
    chunk_overlap: int,
    retriever_k: int,
) -> Tuple[Any, Any, Any]:
    """从 Wiki 页面构建检索工具。

    Args:
        wiki_url: 目标网页地址。
        embedding_model: 向量模型名。
        chunk_size: 切块大小。
        chunk_overlap: 块重叠长度。
        retriever_k: 检索返回条数。

    Returns:
        (retriever_tool, retriever, vectorstore)
    """
    from langchain.tools.retriever import create_retriever_tool
    from langchain_community.document_loaders import WebBaseLoader
    from langchain_community.vectorstores import FAISS
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    docs = WebBaseLoader(wiki_url).load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=['\n\n', '\n', '。', '！', '？', '；', '，', ' ', ''],
    )
    chunks = splitter.split_documents(docs)

    emb = get_embeddings(model=embedding_model)
    vectorstore = FAISS.from_documents(chunks, emb)
    retriever = vectorstore.as_retriever(
        search_type='mmr',
        search_kwargs={'k': retriever_k, 'fetch_k': max(16, retriever_k * 4), 'lambda_mult': 0.6},
    )

    retriever_tool = create_retriever_tool(
        retriever=retriever,
        name='wiki_search',
        description='问题与猫的知识相关时优先调用该工具。',
    )
    return retriever_tool, retriever, vectorstore

web_tool = build_web_search_tool(max_results=cfg.tavily_max_results)
wiki_tool, wiki_retriever, wiki_vs = build_wiki_retriever_tool(
    wiki_url=cfg.wiki_url,
    embedding_model=cfg.embedding_model,
    chunk_size=cfg.chunk_size,
    chunk_overlap=cfg.chunk_overlap,
    retriever_k=cfg.retriever_k,
)
TOOLS = [web_tool, wiki_tool]
print('tools:', [getattr(t, 'name', type(t).__name__) for t in TOOLS])

In [ ]:
def preview_hits(retriever, query: str, n: int = 3, max_chars: int = 240) -> None:
    """打印检索命中，观察召回质量。"""
    docs = retriever.invoke(query)
    print('hits:', len(docs))
    for i, d in enumerate(docs[:n]):
        print(f'\n--- hit[{i}] ---')
        print(d.page_content[:max_chars].replace('\n', '\\n'))
        print('metadata:', d.metadata)

def simple_retrieval_eval(retriever, samples):
    """一个非常轻量的检索评估函数。

    Args:
        retriever: 检索器。
        samples: [(query, keyword), ...]，命中块中包含 keyword 视为成功。
    """
    hit = 0
    for query, keyword in samples:
        docs = retriever.invoke(query)
        text = '\n'.join(d.page_content for d in docs)
        ok = keyword in text
        hit += int(ok)
        print(f'query={query} | keyword={keyword} | hit={ok}')
    print('recall@k(rough):', f'{hit}/{len(samples)}')

preview_hits(wiki_retriever, '猫有哪些典型特征？')
samples = [
    ('猫有什么生活习性？', '猫'),
    ('家猫和野猫有何不同？', '猫'),
]
simple_retrieval_eval(wiki_retriever, samples)

## 3. 创建 Agent 并执行

这里采用 `create_tool_calling_agent`。为了调试方便，开启 `return_intermediate_steps=True`。

In [ ]:
from __future__ import annotations

import os
from typing import Any, Iterable, Optional

def build_agent_executor(*, tools, model: str, temperature: float = 0.1, verbose: bool = True) -> Optional[Any]:
    """构建 AgentExecutor。"""
    if not os.getenv('OPENAI_API_KEY'):
        print('缺少 OPENAI_API_KEY，跳过 Agent 创建。')
        return None
    from langchain import hub
    from langchain.agents import AgentExecutor, create_tool_calling_agent
    from langchain_openai import ChatOpenAI

    prompt = hub.pull('hwchase17/openai-functions-agent')
    llm = ChatOpenAI(model=model, temperature=temperature)
    agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
    return AgentExecutor(
        agent=agent,
        tools=tools,
        verbose=verbose,
        return_intermediate_steps=True,
    )

def run_agent_once(agent_executor, query: str):
    """执行一次 Agent，并打印中间步骤。"""
    if agent_executor is None:
        print('agent_executor 不可用。')
        return None
    result = agent_executor.invoke({'input': query})
    print('\n=== output ===')
    print(result.get('output'))
    print('\n=== intermediate_steps ===')
    for i, step in enumerate(result.get('intermediate_steps', [])):
        action, obs = step
        print(f'[{i}] tool={getattr(action, "tool", "")}')
        print('obs:', str(obs)[:220])
    return result

def run_batch_queries(agent_executor, queries: Iterable[str]):
    """批量执行多个问题，便于 smoke test。"""
    outputs = []
    for q in queries:
        print('\n' + '=' * 30)
        print('Q:', q)
        res = run_agent_once(agent_executor, q)
        outputs.append(res)
    return outputs

agent_executor = build_agent_executor(
    tools=TOOLS,
    model=cfg.chat_model,
    temperature=cfg.temperature,
)
batch_queries = ['猫的特征是什么？', '今天上海天气怎么样？']
_ = run_batch_queries(agent_executor, batch_queries)

## 4. 添加会话记忆（Memory）

同一个 `session_id` 共享历史，不同 `session_id` 自动隔离。

In [ ]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple

def build_memory_agent(agent_executor) -> Tuple[Optional[Any], Dict[str, Any]]:
    """包装带记忆的 Agent。"""
    if agent_executor is None:
        print('agent_executor 不可用，无法添加记忆。')
        return None, {}
    from langchain_community.chat_message_histories import ChatMessageHistory
    from langchain_core.chat_history import BaseChatMessageHistory
    from langchain_core.runnables.history import RunnableWithMessageHistory

    store: Dict[str, ChatMessageHistory] = {}

    def get_session_history(session_id: str) -> BaseChatMessageHistory:
        if session_id not in store:
            store[session_id] = ChatMessageHistory()
        return store[session_id]

    wrapped = RunnableWithMessageHistory(
        runnable=agent_executor,
        get_session_history=get_session_history,
        input_messages_key='input',
        history_messages_key='chat_history',
    )
    return wrapped, store

def memory_chat(agent_with_history, *, query: str, session_id: str):
    """带记忆执行问答。"""
    if agent_with_history is None:
        print('agent_with_history 不可用。')
        return None
    result = agent_with_history.invoke(
        {'input': query},
        config={'configurable': {'session_id': session_id}},
    )
    print(f'[session={session_id}] Q: {query}')
    print('A:', result.get('output'))
    return result

agent_with_history, memory_store = build_memory_agent(agent_executor)
_ = memory_chat(agent_with_history, query='你好，我叫Cyber', session_id='u1')
_ = memory_chat(agent_with_history, query='我叫什么名字？', session_id='u1')
_ = memory_chat(agent_with_history, query='我叫什么名字？', session_id='u2')

## 5. 进阶优化：缓存与重试（可选）

下面给出一个轻量示例：
- `CachedEmbeddings`：减少重复 embedding 计算
- `invoke_with_retry`：模型调用失败时自动重试

这段代码是可复用模板，你可以在项目里直接扩展。

In [ ]:
from __future__ import annotations

import time
from typing import Any, Dict, List

class CachedEmbeddings:
    """对任意 Embeddings 的轻量缓存包装。

    用法：
        cached = CachedEmbeddings(base_embeddings)
        vec1 = cached.embed_query('什么是RAG')
        vec2 = cached.embed_query('什么是RAG')  # 第二次命中缓存
    """

    def __init__(self, base_embeddings: Any):
        self.base = base_embeddings
        self.query_cache: Dict[str, List[float]] = {}
        self.doc_cache: Dict[str, List[float]] = {}

    def embed_query(self, text: str) -> List[float]:
        if text in self.query_cache:
            return self.query_cache[text]
        vec = self.base.embed_query(text)
        self.query_cache[text] = vec
        return vec

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        result: List[List[float]] = []
        miss_idx: List[int] = []
        miss_texts: List[str] = []
        for i, text in enumerate(texts):
            if text in self.doc_cache:
                result.append(self.doc_cache[text])
            else:
                result.append([])
                miss_idx.append(i)
                miss_texts.append(text)
        if miss_texts:
            new_vecs = self.base.embed_documents(miss_texts)
            for i, text, vec in zip(miss_idx, miss_texts, new_vecs):
                self.doc_cache[text] = vec
                result[i] = vec
        return result

def invoke_with_retry(runnable: Any, payload: Dict[str, Any], max_retries: int = 3, sleep_s: float = 1.0):
    """统一重试包装。

    Args:
        runnable: 具有 invoke 方法的对象。
        payload: invoke 参数。
        max_retries: 最大重试次数。
        sleep_s: 每次失败后的等待秒数。
    """
    last_err = None
    for i in range(max_retries):
        try:
            return runnable.invoke(payload)
        except Exception as exc:
            last_err = exc
            print(f'retry {i+1}/{max_retries} failed:', exc)
            time.sleep(sleep_s)
    raise RuntimeError(f'invoke failed after {max_retries} retries') from last_err

# 示例：仅当 agent_executor 可用时演示重试调用
if agent_executor is not None:
    try:
        demo = invoke_with_retry(agent_executor, {'input': '请简要总结猫的特征'})
        print('retry invoke output:', demo.get('output'))
    except Exception as e:
        print('retry demo failed:', e)

## 6. 工程化落地建议

1. 检索质量：加入 rerank 或混合检索（向量 + BM25）。
2. 稳定性：对外部 API 增加超时、重试、熔断。
3. 成本控制：embedding 缓存、上下文压缩、控制 top-k。
4. 可观测性：记录 query、召回文档、工具调用轨迹、最终答案。
5. 评估回归：准备固定问题集，参数调整后自动回归。

## Optimization Add-on: retry wrapper + smoke tests

In [ ]:
import time
from typing import Any, Dict, Iterable
def invoke_with_retry(runnable: Any, payload: Dict[str, Any], retries: int=3, sleep_s: float=1.0):
    last=None
    for i in range(retries):
        try: return runnable.invoke(payload)
        except Exception as e:
            last=e
            print(f'retry {i+1}/{retries} failed:', e)
            time.sleep(sleep_s)
    raise RuntimeError('invoke failed') from last
def smoke(executor, queries: Iterable[str]):
    if executor is None: print('executor unavailable'); return
    for q in queries:
        print('\nQ:', q)
        try:
            res=invoke_with_retry(executor, {'input': q})
            print('A:', res.get('output'))
        except Exception as e:
            print('failed', e)
smoke(globals().get('agent_executor'), ['???????', '??????????'])

## 深度笔记：综合案例的工程化拆解

一个可上线的 Agent，不只是“能回答问题”，还需要：
- 稳定性：失败可重试
- 可观测：知道用了哪个工具、检索了哪些内容
- 可维护：参数集中配置、模块解耦

In [ ]:
import time
from typing import Any, Dict, Iterable

def invoke_with_retry(runnable: Any, payload: Dict[str, Any], retries: int = 3, sleep_s: float = 1.0):
    """统一重试包装。

    适用场景：
    - LLM API 短暂超时
    - 工具调用网络抖动
    """
    last_err = None
    for i in range(retries):
        try:
            return runnable.invoke(payload)
        except Exception as exc:
            last_err = exc
            print(f'retry {i+1}/{retries} failed:', exc)
            time.sleep(sleep_s)
    raise RuntimeError('invoke failed after retries') from last_err

def smoke_test(executor, queries: Iterable[str]):
    """批量 smoke test：快速检查主链路是否可用。"""
    if executor is None:
        print('executor 不可用，跳过。')
        return
    for q in queries:
        print('\n' + '=' * 30)
        print('Q:', q)
        try:
            result = invoke_with_retry(executor, {'input': q})
            print('A:', result.get('output'))
        except Exception as e:
            print('failed:', e)

if 'agent_executor' in globals():
    smoke_test(agent_executor, ['猫有哪些特征？', '今天上海天气怎么样？'])
else:
    print('请先运行构建 agent_executor 的单元。')

### 学习路径建议

1. 先跑通“无记忆”版本，确认工具路由逻辑。
2. 再启用记忆，观察不同 `session_id` 的差异。
3. 最后尝试加入你的业务数据，完成一个最小可用问答系统。